# Financial Transaction Analytics\nSynthetic portfolio project demonstrating Python/pandas validation, KPI analysis, trends, and anomaly flags.

In [ ]:
import pandas as pd\nfrom pathlib import Path\ndata_path = Path('../data/transactions.csv')\ndf = pd.read_csv(data_path, parse_dates=['transaction_date'])\ndf.head()

## Data validation\nCheck required fields, duplicates, missing values, invalid amounts, and allowed statuses.

In [ ]:
required = ['transaction_id','transaction_date','account_id','transaction_type','channel','amount','status','region']\nprint('Missing required columns:', [c for c in required if c not in df.columns])\nprint('Duplicate transaction IDs:', df['transaction_id'].duplicated().sum())\nprint('Missing values:', int(df.isna().sum().sum()))\nprint('Non-positive amounts:', int((df['amount'] <= 0).sum()))

## KPI summary

In [ ]:
completed = df[df['status']=='Completed']\nkpis = {\n 'Total Transactions': len(df),\n 'Completed Transactions': len(completed),\n 'Completed Value': round(completed['amount'].sum(),2),\n 'Average Completed Amount': round(completed['amount'].mean(),2),\n 'Declined Transactions': int((df['status']=='Declined').sum()),\n 'Pending Transactions': int((df['status']=='Pending').sum())\n}\npd.Series(kpis, name='Value')

## Monthly and channel analysis

In [ ]:
monthly = completed.assign(month=completed['transaction_date'].dt.to_period('M').astype(str)).groupby('month').agg(transaction_count=('transaction_id','count'), transaction_value=('amount','sum')).reset_index()\nmonthly

In [ ]:
channel = df.groupby('channel').agg(transaction_count=('transaction_id','count'), average_amount=('amount','mean')).round(2).sort_values('transaction_count', ascending=False)\nchannel

## Analytical anomaly flag\nFor portfolio demonstration, an IQR rule flags unusually high amounts for review. A flag is not evidence of fraud.

In [ ]:
q1, q3 = df['amount'].quantile([0.25,0.75])\niqr = q3-q1\nthreshold = q3 + 1.5*iqr\ndf['high_amount_flag'] = df['amount'] > threshold\nprint('IQR threshold:', round(threshold,2))\ndf.loc[df['high_amount_flag'], ['transaction_id','transaction_date','account_id','amount','channel','status']].sort_values('amount', ascending=False).head(15)

## Business interpretation\nQuestions for follow-up include: Are declines concentrated in a channel? Are unusual amounts expected for the affected accounts? Do pending transactions show a recurring operational pattern? The next step would be to validate these observations with business owners before recommending process changes.